# 3.0 Fruiting-body image EDA (Harvest readiness)

**Question.** Before more Kaggle training: how do the photo sets behind **Harvest readiness** differ from farm to farm, in size, quality, light, colour and background? Run 2 scored 0.87 to 0.90 on the same farm but 0.00 on the other. Run 3 asks whether farms look too different. This notebook looks at the photos only. **No YOLO training, CPU is enough.**

**Photos.** The same five sets as `3.2.kaggle_harvest_readiness.ipynb` (run 3): Duman (Mendeley hf55tkx489, uploaded to Kaggle) and the Roboflow sets angelo, amr-asmr, thesis, ros2, plus **abhi** (the pink-oyster set of run 1, an older notebook now only in git history, which run 3 left out; it is here so pink and white oysters can be compared). A set that cannot be downloaded is skipped and named. The download, label-mapping and perceptual-hash code is copied from run 3 (not imported); small additions are marked `# added`. One copy per source photo is kept (Roboflow augmented copies are dropped).

| Section | Plain question |
|---|---|
| 1 Inventory | How many photos, boxes, young vs ready, which sizes, how many duplicates, per farm? |
| 2 Photo quality | Are some farms blurrier, more blown-out or darker? |
| 3 Lighting and colour | Do farms differ in brightness and colour? Do pink and white oysters stand out? |
| 4 Background vs class | Could a model tell young from ready by the background or the camera distance instead of the mushroom? |
| 5 Farm guesser | Can a tiny model tell which farm a photo came from? High accuracy = farms look very different = cross-farm failure is expected. |
| 6 Biological variation | Which species or colour type and which growth stages does each farm show? |
| 7 Summary | One row per farm, to read by screenshot. |

**Before you press Run All (once):**
1. Duman must be a private Kaggle dataset with the raw photos and the YOLO label files (see the box below).
2. Right panel, *Session options*: Accelerator = **None** (CPU is enough), Internet = **On**.
3. *Add-ons, Secrets*: switch `ROBOFLOW_API_KEY` on for this notebook. The key is never printed or saved.
4. *Add Input*: add your Duman dataset.

Nothing is written except temporary photo folders in `/kaggle/working`, which are deleted at the end. No CSV files are produced.

### Uploading Duman to Kaggle (same dataset as run 3)
On kaggle.com: **Create, New Dataset**, title `duman-oyster-maturity`, keep it **Private**, with the two folders from `mycodemo/data/external/d010_duman2024/`: `Raw Images` (555 photos) and `Maturity Labelled Images and Annotation Files/YOLO Format`. The notebook finds the folders by what is inside them. If you already made this dataset for run 3, just add it as input.

In [ ]:
# 1. Install the libraries we need (Kaggle has the rest, including opencv and scikit-learn)
!pip install -q roboflow imagehash

In [ ]:
# 2. Settings (copied from run 3; the training settings are left out)
import re
def norm(s):
    # match label names ignoring case, spaces, dashes and underscores
    return re.sub(r"[\s_\-]+", "", str(s).lower())

# farm: (Roboflow workspace, project, {their label: our label})
ROBOFLOW = {
    "angelo": ("angelo-neyra-9rg4y", "mushroom-w7ucu",
               {"not_ready": "young", "ready": "ready", "overdue": "ready"}),
    # added: the pink-oyster set of run 1 (older notebook, in git history), kept here for the colour comparison only
    "abhi":   ("abhi-sok0u", "oyster-dntlz",
               {"pinning": "young", "early growth": "young", "mature": "ready"}),
    "amr":    ("amr-asmr", "oyster-mushroom-growth",
               {"juvenile": "young", "intermediate": "young", "harvest": "ready"}),
    "thesis": ("thesis-fgp1n", "oyster-mushroom-v2",
               {"not ready": "young", "ready": "ready"}),
    "ros2":   ("ros2-plan", "oyster-mushroom-bkezg",
               {"immature": "young", "mature": "ready"}),
}
DUMAN_LABELS = {"immature": "young", "mature": "ready"}
MAPPING = {"duman": DUMAN_LABELS, **{k: v[2] for k, v in ROBOFLOW.items()}}
MAPPING = {f: {norm(k): v for k, v in m.items()} for f, m in MAPPING.items()}

OUR_LABELS = ["young", "ready"]   # class 0 = young, class 1 = ready
HASH_DIST = 4                     # perceptual-hash distance (of 64 bits) counted as "near-identical"
SEED = 0
WORK = "/kaggle/working"

In [ ]:
# 3. Find the Duman folders you uploaded (by content, not by folder name)
import os, glob, collections
PHOTO_EXT = (".jpg", ".jpeg")

def list_images(d):
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(PHOTO_EXT))

def show_tree():
    for root, dirs, files in sorted(os.walk("/kaggle/input")):
        kinds = collections.Counter(os.path.splitext(f)[1].lower() or "(no extension)" for f in files)
        depth = root.count(os.sep) - 2
        print("  " * depth + os.path.basename(root) + "/", dict(kinds) if kinds else "")

def find_dir():
    best_photos, best_labels = (0, None), (0, None)
    for root, dirs, files in os.walk("/kaggle/input"):
        n_img = sum(f.lower().endswith(PHOTO_EXT) for f in files)
        if n_img > best_photos[0]:
            best_photos = (n_img, root)
        if "classes.txt" in files:
            names = [n.lower() for n in open(os.path.join(root, "classes.txt"), errors="ignore").read().split()]
            n_txt = sum(f.lower().endswith(".txt") for f in files)
            # the Duman maturity labels list Mature and Immature; the other YOLO folder does not
            if "mature" in names and "immature" in names and n_txt > best_labels[0]:
                best_labels = (n_txt, root)
    if best_photos[1] is None or best_labels[0] < 10:
        print("Could not find the Duman photos and labels. What is under /kaggle/input:")
        show_tree()
        raise FileNotFoundError("Duman photos and/or labels not found under /kaggle/input. "
                                "Attach the Duman dataset to this notebook with Add Input (right panel), then run again.")
    return best_photos[1], best_labels[1]

DUMAN_SKIPPED = None      # added: a missing Duman input skips Duman and names it, like a failed Roboflow set
try:
    DUMAN_IMG, DUMAN_LAB = find_dir()
    print("photos:", DUMAN_IMG, len(list_images(DUMAN_IMG)))
    print("labels:", DUMAN_LAB, len(glob.glob(DUMAN_LAB + "/*.txt")))
    print("their label names:", open(DUMAN_LAB + "/classes.txt").read().split())
except FileNotFoundError as e:
    DUMAN_IMG = DUMAN_LAB = None
    DUMAN_SKIPPED = "FileNotFoundError: " + str(e)[:160]
    print("SKIPPED duman:", DUMAN_SKIPPED)

In [ ]:
# 4. Download the Roboflow sets (version with the fewest photos = no flipped/augmented copies)
#    A set that fails is skipped and named; the key is never printed.
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=KEY)
DOWNLOADS, SKIPPED = {}, {}
for short, (ws, slug, _) in ROBOFLOW.items():
    try:
        proj = rf.workspace(ws).project(slug)
        vers = []
        for v in proj.versions():
            n = getattr(v, "images", None)
            if not n:
                splits = getattr(v, "splits", None) or {}
                n = sum(int(x) for x in splits.values()) if splits else 10**9
            vers.append((int(n), int(str(v.version).split("/")[-1])))
        ver = sorted(vers)[0][1]
        DOWNLOADS[short] = proj.version(ver).download("yolov11", location=f"{WORK}/raw_{short}", overwrite=True).location
        print(f"{short}: versions (photos, version) {sorted(vers)} -> chose version {ver}")
    except Exception as e:
        SKIPPED[short] = f"{type(e).__name__}: {str(e).replace(KEY, '***')[:200]}"
        print(f"SKIPPED {short} ({ws}/{slug}): {SKIPPED[short]}")
print("\nfarms downloaded:", list(DOWNLOADS), "| skipped:", list(SKIPPED))

## Put all farms into the same shape (copied from run 3)
Every label is rewritten as 0 = young, 1 = ready (names matched ignoring case, spaces, dashes, underscores). Traced outlines become the smallest rectangle around them. Labels not in the mapping are dropped and counted. **Added:** every original label name is also counted (section 6 shows them), and Roboflow augmented copies (`<name>.rf.<code>.jpg`, same text before `.rf.`) are dropped so one copy per source photo is kept.

In [ ]:
import shutil, random, yaml, collections, glob, datetime
import numpy as np, pandas as pd
from PIL import Image

def to_box(parts):
    # 'class x y w h' stays; 'class x1 y1 x2 y2 ...' (outline) becomes its surrounding rectangle
    nums = [float(p) for p in parts[1:]]
    if len(nums) == 4:
        return nums
    xs, ys = nums[0::2], nums[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0]

ORIG = {}   # pooled photo path -> original file name (Duman: the timestamp)
THEIR = {}  # added: farm -> how often each ORIGINAL label name occurs (before mapping)
AUG = collections.Counter()   # added: farm -> Roboflow augmented copies dropped

def pool(short, pairs, their, mapping):
    # pairs = list of (image path, label path); their = list of their label names
    new_id = {i: OUR_LABELS.index(mapping[norm(n)]) for i, n in enumerate(their) if norm(n) in mapping}
    out = f"{WORK}/pool_{short}"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(f"{out}/images"); os.makedirs(f"{out}/labels")
    counts, dropped = collections.Counter(), collections.Counter()
    their_counts = collections.Counter()   # added
    for k, (img, lab) in enumerate(pairs):
        lines = []
        if lab and os.path.exists(lab):
            for line in open(lab):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                their_counts[their[c] if c < len(their) else f"unknown class {c}"] += 1   # added
                if c in new_id:
                    lines.append(" ".join([str(new_id[c])] + [f"{v:.6f}" for v in to_box(parts)]))
                    counts[OUR_LABELS[new_id[c]]] += 1
                else:
                    dropped[their[c] if c < len(their) else f"unknown class {c}"] += 1
        name = f"{short}_{k:05d}"
        dst = f"{out}/images/{name}{os.path.splitext(img)[1]}"
        shutil.copy(img, dst)
        ORIG[dst] = os.path.basename(img)
        open(f"{out}/labels/{name}.txt", "w").write("\n".join(lines))
    print(f"{short}: {len(pairs)} photos, clusters {dict(counts)}, DROPPED labels {dict(dropped)}, "
          f"{sum(counts.values()) / max(len(pairs), 1):.1f} clusters per photo")
    THEIR[short] = their_counts   # added
    return out

POOL = {}
# duman: one label file per raw photo, same file name
if DUMAN_IMG:
    dum_names = open(DUMAN_LAB + "/classes.txt").read().split()
    POOL["duman"] = pool("duman", [(img, f"{DUMAN_LAB}/{os.path.splitext(os.path.basename(img))[0]}.txt")
                                   for img in list_images(DUMAN_IMG)], dum_names, MAPPING["duman"])
else:
    SKIPPED["duman"] = DUMAN_SKIPPED

# Roboflow farms: pool train/valid/test together (the held-out farm is scored on everything anyway)
for short, dl in DOWNLOADS.items():
    names = yaml.safe_load(open(f"{dl}/data.yaml"))["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    pairs, seen = [], set()
    for split in ["train", "valid", "test"]:
        for img in sorted(glob.glob(f"{dl}/{split}/images/*")):
            key = os.path.basename(img).split(".rf.")[0]       # added: copies share the text before ".rf."
            if key in seen:
                AUG[short] += 1
                continue
            seen.add(key)
            base = os.path.splitext(os.path.basename(img))[0]
            pairs.append((img, f"{dl}/{split}/labels/{base}.txt"))
    POOL[short] = pool(short, pairs, names, MAPPING[short])
assert POOL, "no photo set could be loaded: " + str(SKIPPED)
print("\nRoboflow augmented copies dropped:", dict(AUG) or "none")

## Groups and duplicates (copied from run 3)
* **Duman:** photos from the same calendar day (UTC, from the millisecond time stamp in the file name) form one group.
* **Other farms:** photos whose perceptual hashes differ by at most `HASH_DIST` bits form one group.
* **Across farms:** a photo that is near-identical to a photo of an earlier farm (Duman first) is removed.

Splits always move whole groups.

**Added:** for every farm, including Duman, the perceptual-hash groups are also kept (`nn_group`) so section 1 can count near-duplicate groups. The `group` column is the run 3 rule and is what the cross-validation below splits by.

In [ ]:
import imagehash

def phash_int(path):
    with Image.open(path) as im:
        h = imagehash.phash(im.convert("RGB")).hash.flatten()
    return int(np.packbits(h).view(">u8")[0])

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def dist_to_all(H, h):
    # Hamming distance from hash h to every hash in H
    return POP[(H ^ np.uint64(h)).view(np.uint8)].reshape(-1, 8).sum(axis=1)

PHOTOS = {}      # farm -> list of (photo path, group id)
NN = {}          # added: photo path -> near-duplicate (perceptual hash) group
kept_hashes = np.zeros(0, dtype=np.uint64)
removed = {}
for short in POOL:                      # duman is first, so its photos win the cross-farm duplicates
    imgs = sorted(glob.glob(f"{POOL[short]}/images/*"))
    H = np.array([phash_int(p) for p in imgs], dtype=np.uint64)
    keep = np.ones(len(imgs), bool)
    if len(kept_hashes):
        for i in range(len(imgs)):
            if dist_to_all(kept_hashes, int(H[i])).min() <= HASH_DIST:
                keep[i] = False
    removed[short] = int((~keep).sum())
    for p in np.array(imgs)[~keep]:
        os.remove(p); os.remove(p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt")
    imgs, H = list(np.array(imgs)[keep]), H[keep]
    # near-duplicate groups by perceptual hash (added for every farm; run 3 used them for the Roboflow farms only)
    parent = list(range(len(imgs)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for i in range(len(imgs)):
        for j in np.nonzero(dist_to_all(H[:i], int(H[i])) <= HASH_DIST)[0] if i else []:
            parent[find(i)] = find(int(j))
    nn = [f"{short}_n{find(i)}" for i in range(len(imgs))]
    if short == "duman":
        def day(p):
            try:
                return str(datetime.datetime.fromtimestamp(int(os.path.splitext(ORIG[p])[0]) / 1000, datetime.timezone.utc).date())
            except ValueError:
                return None
        days = [day(p) for p in imgs]
        if None in days:
            print("WARNING: some Duman file names are not timestamps; those photos form their own groups")
        gid = [f"duman_{d}" if d else f"duman_single_{i}" for i, d in enumerate(days)]
    else:
        gid = nn
    PHOTOS[short] = list(zip(imgs, gid))
    NN.update(zip(imgs, nn))   # added
    kept_hashes = np.concatenate([kept_hashes, H])

print("farm     photos  groups  (removed as copies of an earlier farm)")
for s, items in PHOTOS.items():
    print(f"{s:8s} {len(items):6d} {len(set(g for _, g in items)):7d}  ({removed[s]})")

## Measure every photo once
Each photo is opened once and resized so its long side is 1024 px (small photos are enlarged, so a tiny photo reads as blurry, which is also true for a model). From that copy we take: sharpness (variance of a Laplacian on greyscale; higher = sharper), % blown-out pixels (V > 250), % very dark pixels (V < 15), mean brightness V and mean saturation S (both 0 to 255), red/blue ratio, a hue histogram of the coloured pixels (S and V at least 40), a 3x8-bin colour histogram of the whole photo and of the pixels **outside** the boxes, and how much of the photo the boxes cover. A few minutes on CPU.

In [ ]:
# 5. Helpers: one function measures one photo
import numpy as np, pandas as pd
import cv2
from PIL import Image
LONG_SIDE = 1024      # every photo is resized to this long side before it is measured

def bins8(px):
    # 3 x 8 colour histogram: 8 bins for each of R, G, B; each channel sums to 1 (24 numbers)
    if len(px) == 0:
        return np.full(24, np.nan)
    return np.concatenate([np.bincount(px[:, c] >> 5, minlength=8) / len(px) for c in range(3)])

def measure(path, boxes):
    """boxes = list of (class 0/1, x, y, w, h), as fractions of the photo. Returns numbers for one photo."""
    with Image.open(path) as im:
        W, H = im.size
        exif_rotated = im.getexif().get(274, 1) not in (None, 1)
        rgb = np.asarray(im.convert("RGB"))
    k = LONG_SIDE / max(W, H)
    small = cv2.resize(rgb, (max(1, round(W * k)), max(1, round(H * k))),
                       interpolation=cv2.INTER_AREA if k < 1 else cv2.INTER_LINEAR)
    hh, ww = small.shape[:2]
    grey = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)          # H 0-179, S 0-255, V 0-255
    Hu, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    masks = {0: np.zeros((hh, ww), bool), 1: np.zeros((hh, ww), bool)}
    for c, x, y, w, h in boxes:
        x0, x1 = int(np.clip(round((x - w / 2) * ww), 0, ww)), int(np.clip(round((x + w / 2) * ww), 0, ww))
        y0, y1 = int(np.clip(round((y - h / 2) * hh), 0, hh)), int(np.clip(round((y + h / 2) * hh), 0, hh))
        masks[int(c)][y0:y1, x0:x1] = True
    inside = masks[0] | masks[1]
    mean_rgb = small.reshape(-1, 3).mean(axis=0)
    row = {"W": W, "H": H, "exif_rotated": bool(exif_rotated),
           "n_boxes": len(boxes), "n_0": sum(int(b[0]) == 0 for b in boxes), "n_1": sum(int(b[0]) == 1 for b in boxes),
           "cover": 100 * inside.mean(), "cover_0": 100 * masks[0].mean(), "cover_1": 100 * masks[1].mean(),
           "sharp": float(cv2.Laplacian(grey, cv2.CV_64F).var()),
           "blown": 100 * (V > 250).mean(), "dark": 100 * (V < 15).mean(),
           "meanV": float(V.mean()), "meanS": float(S.mean()),
           "rb": float(mean_rgb[0] / max(mean_rgb[2], 1e-6))}
    if inside.any():                                        # colour of the mushrooms / mould themselves
        px = small[inside]
        row["rb_in"] = float(px[:, 0].mean() / max(px[:, 2].mean(), 1e-6))
        row["S_in"] = float(S[inside].mean())
    else:
        row["rb_in"] = row["S_in"] = np.nan
    out_px = small[~inside]                                 # background: everything outside the boxes
    row["out_pct"] = 100 * (~inside).mean()
    if len(out_px) >= 0.01 * hh * ww:
        row["R_out"], row["G_out"], row["B_out"] = (float(v) for v in out_px.mean(axis=0))
        row["V_out"], row["S_out"] = float(V[~inside].mean()), float(S[~inside].mean())
        hist_out = bins8(out_px)
    else:
        row["R_out"] = row["G_out"] = row["B_out"] = row["V_out"] = row["S_out"] = np.nan
        hist_out = np.full(24, np.nan)
    coloured = (S >= 40) & (V >= 40)                        # grey/white/dark pixels have no useful hue
    hue = np.bincount(Hu[coloured] // 5, minlength=36)[:36] / max(coloured.sum(), 1)   # 36 bins of 10 degrees
    return row, bins8(small.reshape(-1, 3)), hist_out, hue

def thumb(path, size=300):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((size, size))
        return im.copy()

import matplotlib.pyplot as plt

def box_by(ax, df, key, col, order, title, log=False, colours=None):
    # one box per value of `key`, in the given order
    order = [s for s in order if df.loc[df[key] == s, col].notna().any()]
    data = [df.loc[df[key] == s, col].dropna().values for s in order]
    bp = ax.boxplot(data, patch_artist=True, flierprops=dict(markersize=2, alpha=0.4), medianprops=dict(color="black"))
    for patch, s in zip(bp["boxes"], order):
        patch.set_facecolor((colours or {}).get(s, "lightgrey")); patch.set_alpha(0.8)
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order, rotation=20)
    ax.set_title(title, fontsize=10)
    if log:
        ax.set_yscale("log")

def show_thumbs(groups, title):
    # groups = {name: [(photo path, caption), ...]}; one row per name
    n = max(1, max(len(v) for v in groups.values()))
    fig, axes = plt.subplots(len(groups), n, figsize=(3.2 * n, 2.7 * len(groups)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for r, (name, items) in enumerate(groups.items()):
        for c, (p, cap) in enumerate(items):
            axes[r][c].imshow(thumb(p)); axes[r][c].set_title(f"{name}\n{cap}", fontsize=7)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

def group_cv(X, y, groups, proba=False, max_splits=5):
    """Logistic regression scored out-of-fold. Photos of one group never sit on both sides of a split.
    Returns None when there are too few groups per class to split."""
    y, groups = np.asarray(y), np.asarray(groups)
    n_splits = min([max_splits] + [len(set(groups[y == c])) for c in set(y)])
    if n_splits < 2 or len(set(y)) < 2:
        return None
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    try:
        out = cross_val_predict(clf, X, y, cv=cv, groups=groups, method="predict_proba" if proba else "predict")
    except ValueError:
        return None
    return out[:, 1] if proba else out

In [ ]:
# 6. Measure all photos (about 1 to 2 minutes per 1000 photos)
FARMS = list(PHOTOS)
FCOL = {f: plt.cm.tab10(i) for i, f in enumerate(FARMS)}

def pooled_boxes(img):
    lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    rows = [l.split() for l in open(lab).read().splitlines() if l.strip()]
    return [(int(c), float(x), float(y), float(w), float(h)) for c, x, y, w, h in rows]

rows, H_ALL, H_OUT, H_HUE = [], [], [], []
for farm in FARMS:
    for p, g in PHOTOS[farm]:
        r, ha, ho, hu = measure(p, pooled_boxes(p))
        r.update(farm=farm, path=p, group=g, nn_group=NN[p])
        r["kind"] = ("none" if r["n_boxes"] == 0 else "young" if r["n_1"] == 0 else "ready" if r["n_0"] == 0 else "mixed")
        rows.append(r); H_ALL.append(ha); H_OUT.append(ho); H_HUE.append(hu)
    print(f"{farm}: {len(PHOTOS[farm])} photos measured")
PH = pd.DataFrame(rows)
H_ALL, H_OUT, H_HUE = np.array(H_ALL), np.array(H_OUT), np.array(H_HUE)
PH["out_ok"] = np.isfinite(H_OUT).all(axis=1)       # False when the boxes cover (almost) the whole photo
print("\nphotos in the table:", len(PH))

## 1. Inventory per farm
**Question: what is in each farm's set?** Photos after de-duplication, near-duplicate groups (perceptual hash, 2 or more photos), boxes per photo, young vs ready boxes (class distribution) and photo sizes. Very different sizes between farms, or one label dominating, can explain a bad score on their own.

In [ ]:
# 7. Inventory table (farms are the columns) and class distribution
from collections import Counter
rows = {}
for f in FARMS:
    d = PH[PH.farm == f]
    grp = d.nn_group.value_counts()
    ny, nr = int(d.n_0.sum()), int(d.n_1.sum())
    top = ", ".join(f"{w}x{h} ({n})" for (w, h), n in Counter(zip(d.W, d.H)).most_common(3))
    rows[f] = {"photos": len(d),
               "Roboflow augmented copies dropped": AUG.get(f, 0),
               "copies of an earlier farm removed": removed[f],
               "near-duplicate groups (2+ photos)": int((grp >= 2).sum()),
               "photos inside those groups": int(grp[grp >= 2].sum()),
               "groups used for splits (run 3 rule)": d.group.nunique(),
               "boxes": int(d.n_boxes.sum()),
               "boxes per photo (mean)": round(d.n_boxes.mean(), 1),
               "boxes per photo (median, max)": f"{d.n_boxes.median():.0f}, {d.n_boxes.max()}",
               "photos with no box": int((d.n_boxes == 0).sum()),
               "young boxes": ny, "ready boxes": nr,
               "young share %": round(100 * ny / max(ny + nr, 1), 1),
               "ready share %": round(100 * nr / max(ny + nr, 1), 1),
               "distinct photo sizes": d[["W", "H"]].drop_duplicates().shape[0],
               "most common sizes WxH (photos)": top,
               "EXIF-rotated photos": int(d.exif_rotated.sum())}
INV = pd.DataFrame(rows)
display(INV)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
share = INV.loc[["young share %", "ready share %"]].T.astype(float)
share.plot(kind="barh", stacked=True, ax=axes[0], color=["tab:blue", "tab:orange"])
axes[0].set_title("young vs ready (share of boxes, %)", fontsize=10); axes[0].legend(["young", "ready"], loc="lower right")
INV.loc["photos"].astype(int).plot(kind="barh", ax=axes[1], color=[FCOL[f] for f in FARMS])
axes[1].set_title("photos per farm", fontsize=10)
plt.tight_layout(); plt.show()
if SKIPPED:
    print("SKIPPED sets:", SKIPPED)

**Read it.** Different photo sizes mean the camera or the export differs; 640x640 usually means Roboflow resized the photos, so the original camera distance is hidden in the box sizes (section 4). A share near 100% for one label means the model can score well by guessing that label.

## 2. Photo quality per farm
**Question: are some farms' photos blurrier, more blown-out or darker?** Boxplots per farm, then the 4 worst photos per farm. "Worst" = the photo whose most extreme measure ranks highest among all photos: lowest sharpness, most blown-out pixels or most very dark pixels (the caption says which). Sharpness is on a log scale.

In [ ]:
# 8. Quality boxplots and the 4 worst photos per farm
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
box_by(axes[0], PH, "farm", "sharp", FARMS, "sharpness (Laplacian variance, log scale; higher = sharper)", log=True, colours=FCOL)
box_by(axes[1], PH, "farm", "blown", FARMS, "% blown-out pixels (V > 250)", colours=FCOL)
box_by(axes[2], PH, "farm", "dark", FARMS, "% very dark pixels (V < 15)", colours=FCOL)
plt.tight_layout(); plt.show()

display(PH.groupby("farm")[["sharp", "blown", "dark"]].median().round(2).loc[FARMS]
        .rename(columns={"sharp": "median sharpness", "blown": "median % blown", "dark": "median % dark"}))

# badness = highest rank (0 to 1, among ALL photos) of: blurry, blown out, dark
PH["bad_blur"] = 1 - PH.sharp.rank(pct=True)
PH["bad_blown"] = PH.blown.rank(pct=True)
PH["bad_dark"] = PH.dark.rank(pct=True)
PH["bad"] = PH[["bad_blur", "bad_blown", "bad_dark"]].max(axis=1)
why = {"bad_blur": "blurry", "bad_blown": "blown out", "bad_dark": "dark"}
groups = {}
for f in FARMS:
    items = []
    for _, r in PH[PH.farm == f].nlargest(4, "bad").iterrows():
        reason = why[r[["bad_blur", "bad_blown", "bad_dark"]].astype(float).idxmax()]
        items.append((r.path, f"{reason}: sharp {r.sharp:.0f}, blown {r.blown:.1f}%, dark {r.dark:.1f}%"))
    groups[f] = items
show_thumbs(groups, "4 worst photos per farm (by the measure that ranks worst)")

## 3. Lighting and colour per farm
**Question: do farms differ in brightness and colour, and do pink (P. djamor) and white oysters stand out?** Mean brightness (V) and saturation (S) are 0 to 255. The red/blue ratio is mean red divided by mean blue: around 1 for white and grey, above 1 for warm or pink tones. "inside boxes" looks only at the pixels in the boxes, the mushrooms themselves, so the substrate and the walls do not hide the colour of the fruit. The hue histogram uses coloured pixels only; the shaded band is pink/magenta (300 to 350 degrees).

In [ ]:
# 9. Brightness, saturation, red/blue, hue histogram, and the pink-vs-white scatter
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
box_by(axes[0], PH, "farm", "meanV", FARMS, "mean brightness V (0-255)", colours=FCOL)
box_by(axes[1], PH, "farm", "meanS", FARMS, "mean saturation S (0-255)", colours=FCOL)
box_by(axes[2], PH, "farm", "rb", FARMS, "red/blue ratio, whole photo", colours=FCOL)
box_by(axes[3], PH, "farm", "rb_in", FARMS, "red/blue ratio, inside boxes", colours=FCOL)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
deg = np.arange(36) * 10 + 5
for f in FARMS:
    axes[0].plot(deg, H_HUE[(PH.farm == f).values].mean(axis=0), label=f, color=FCOL[f], lw=2)
axes[0].axvspan(300, 350, color="pink", alpha=0.35)
axes[0].set_xlabel("hue (degrees: 0 red, 60 yellow, 120 green, 240 blue, 300 magenta)")
axes[0].set_ylabel("share of coloured pixels"); axes[0].set_title("hue histogram per farm", fontsize=10); axes[0].legend()
for f in FARMS:
    d = PH[PH.farm == f]
    axes[1].scatter(d.rb_in, d.S_in, s=8, alpha=0.4, color=FCOL[f], label=f)
axes[1].axvline(1, color="grey", lw=0.8, ls=":")
axes[1].set_xlabel("red/blue ratio inside boxes (1 = no red tint)"); axes[1].set_ylabel("mean saturation inside boxes")
axes[1].set_title("mushroom colour per photo: white oysters bottom left, pink up and right", fontsize=10)
axes[1].legend(markerscale=2)
plt.tight_layout(); plt.show()

pink = H_HUE[:, 30:35].sum(axis=1)          # hue bins 300 to 350 degrees
PH["pink_share"] = 100 * pink
display(PH.groupby("farm")[["meanV", "meanS", "rb", "rb_in", "S_in", "pink_share"]].median().round(2).loc[FARMS]
        .rename(columns={"meanV": "median V", "meanS": "median S", "rb": "median R/B", "rb_in": "median R/B in boxes",
                         "S_in": "median S in boxes", "pink_share": "median % of coloured pixels pink"}))

**Read it.** A farm whose dots sit to the right and up in the scatter has pink or strongly coloured mushrooms; farms bottom left have white or pale ones. If one farm stands alone in colour, a model trained on the other farms has never seen that colour, which can explain a 0.00 cross-farm score. Cluster boxes that include a lot of substrate blur this view, so check the thumbnails in section 6.

## 4. Background vs class
**Question: could a model tell young from ready by the background or the camera distance instead of the mushroom?** Per photo we take (a) the % of the photo covered by boxes (large = close-up, small = far shot) and (b) the colour of the pixels **outside** the boxes. Photos are split into young only, ready only or mixed. Then a small logistic regression is asked to separate young-only from ready-only photos using **only** the background colour (3x8 histogram of the pixels outside the boxes), the % covered and the photo size, with no mushroom pixels. It is scored out-of-fold with whole groups held out. AUC 0.5 = no shortcut; clearly above 0.5 means the background or distance carries the label.

In [ ]:
# 10. Cover and background colour by farm and by young/ready
B = PH[PH.n_boxes > 0].copy()
B["cover_pct"] = B.cover.clip(lower=0.01)
KINDS = ["young", "ready", "mixed"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
box_by(axes[0], B, "farm", "cover_pct", FARMS, "% of photo covered by boxes (log scale; large = close-up)", log=True, colours=FCOL)
# grouped: per farm, one box per kind
pos, data, cols, ticks = 1, [], [], []
kc = {"young": "tab:blue", "ready": "tab:orange", "mixed": "tab:green"}
for f in FARMS:
    for k in KINDS:
        v = B.loc[(B.farm == f) & (B.kind == k), "cover_pct"].values
        if len(v):
            data.append(v); cols.append(kc[k]); ticks.append((pos, f"{f}\n{k}"))
            pos += 1
    pos += 1
bp = axes[1].boxplot(data, positions=[t[0] for t in ticks], patch_artist=True, widths=0.8,
                     flierprops=dict(markersize=2, alpha=0.4), medianprops=dict(color="black"))
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c); patch.set_alpha(0.8)
axes[1].set_xticks([t[0] for t in ticks]); axes[1].set_xticklabels([t[1] for t in ticks], fontsize=6, rotation=90)
axes[1].set_yscale("log"); axes[1].set_title("same, split by young-only / ready-only / mixed photos", fontsize=10)
plt.tight_layout(); plt.show()

# background colour outside the boxes: median colour per farm and kind
fig, ax = plt.subplots(figsize=(7, 1 + 0.8 * len(FARMS)))
img = np.ones((len(FARMS), len(KINDS), 3))
for i, f in enumerate(FARMS):
    for j, k in enumerate(KINDS):
        d = B[(B.farm == f) & (B.kind == k) & B.R_out.notna()]
        if len(d):
            img[i, j] = d[["R_out", "G_out", "B_out"]].median().values / 255
            ax.text(j, i, f"n={len(d)}\nV {d.V_out.median():.0f}  S {d.S_out.median():.0f}", ha="center", va="center", fontsize=7,
                    color="white" if img[i, j].mean() < 0.5 else "black")
        else:
            ax.text(j, i, "none", ha="center", va="center", fontsize=7)
ax.imshow(img, aspect="auto"); ax.set_xticks(range(len(KINDS))); ax.set_xticklabels(KINDS)
ax.set_yticks(range(len(FARMS))); ax.set_yticklabels(FARMS)
ax.set_title("median colour of the pixels OUTSIDE the boxes", fontsize=10)
plt.tight_layout(); plt.show()

display(B.groupby(["farm", "kind"]).agg(photos=("path", "size"), cover_median=("cover", "median"),
        V_out=("V_out", "median"), S_out=("S_out", "median")).round(1))

# background-only young/ready test, per farm and pooled
def bg_features(d, with_colour=True):
    cols = [np.log(d.cover_pct.values)[:, None], np.log(d[["W", "H"]].values)]
    return np.hstack(([H_OUT[d.index.values]] if with_colour else []) + cols)

BG = {}
test_rows = []
pure = B[B.kind.isin(["young", "ready"]) & B.out_ok]
for name, d in [(f, pure[pure.farm == f]) for f in FARMS] + [("all farms together", pure)]:
    y = (d.kind == "ready").astype(int).values
    row = {"farm": name, "young-only photos": int((y == 0).sum()), "ready-only photos": int((y == 1).sum())}
    if min(row["young-only photos"], row["ready-only photos"]) < 10:
        row["note"] = "too few photos of one label"
    else:
        p_dist = group_cv(np.log(d.cover_pct.values)[:, None], y, d.group.values, proba=True)
        p_all = group_cv(bg_features(d), y, d.group.values, proba=True)
        if p_all is None:
            row["note"] = "too few groups to split"
        else:
            row["AUC, camera distance only"] = round(roc_auc_score(y, p_dist), 2)
            row["AUC, background colour + distance + size"] = round(roc_auc_score(y, p_all), 2)
            row["balanced accuracy, background"] = round(balanced_accuracy_score(y, p_all > 0.5), 2)
            if name in FARMS:
                BG[name] = row["AUC, background colour + distance + size"]
    test_rows.append(row)
display(pd.DataFrame(test_rows).set_index("farm").fillna(""))

**Read it.** Per-farm rows are the clean test. If the AUC is clearly above 0.5 inside a farm, young and ready photos were taken from different distances or in front of different backgrounds, so a model can score well without looking at the mushroom, and the same shortcut will not carry over to another farm. The "all farms together" row also contains farm differences (section 5), so a high value there is expected and says less. A farm with only one label (too few photos of one label) cannot be tested.

## 5. Farm guesser
**Question: can a small model tell which farm a photo came from?** Logistic regression on the 3x8-bin colour histogram of the whole photo plus the photo size (log width, log height), predicting the farm. Scored out-of-fold with whole groups held out (Duman by calendar day, the other farms by near-duplicate group), so near-twins cannot help. The colour-only and size-only rows show which part does the work. **High accuracy means farms look very different, which explains cross-farm failure.** The chance level is shown as "always guess the biggest farm".

In [ ]:
# 11. Farm guesser, scored by group-split cross-validation
if len(FARMS) < 2:
    print("only one farm loaded: nothing to compare")
GUESS = {}
if len(FARMS) >= 2:
    y = PH.farm.values
    X_col, X_size = H_ALL, np.log(PH[["W", "H"]].values)
    variants = {"colour only": X_col, "photo size only": X_size, "colour + size": np.hstack([X_col, X_size])}
    rows, preds = [], {}
    for name, X in variants.items():
        pred = group_cv(X, y, PH.group.values)
        if pred is None:
            print(name, ": too few groups to split"); continue
        preds[name] = pred
        rows.append({"features": name, "accuracy": round(accuracy_score(y, pred), 3),
                     "balanced accuracy": round(balanced_accuracy_score(y, pred), 3)})
    chance = PH.farm.value_counts(normalize=True).max()
    display(pd.DataFrame(rows).set_index("features"))
    print(f"always guessing the biggest farm: accuracy {chance:.3f}; guessing at random: balanced accuracy {1 / len(FARMS):.3f}")
    if "colour + size" in preds:
        conf = pd.crosstab(pd.Series(y, name="true farm"), pd.Series(preds["colour + size"], name="guessed farm")).reindex(index=FARMS, columns=FARMS, fill_value=0)
        print("\nconfusion matrix, colour + size (rows = true farm):")
        display(conf)
        GUESS = {f: round(100 * conf.loc[f, f] / max(conf.loc[f].sum(), 1)) for f in FARMS}
        print("share of each farm's photos guessed right, %:", GUESS)

**Read it.** Balanced accuracy near 0.2 (five farms) would mean the farms look alike. Near 1.0 means a photo's colours and size give away its farm, so a model trained on other farms is looking at photos it has never seen the like of. If "photo size only" is already high, the farms differ in camera or export size, not just in light.

## 6. Biological variation
**Question: which species or colour type and which growth stages does each farm show?** The growth stages are the dataset's own label names, counted before they are mapped to young / ready. The species cannot be read from the files: fill in the notes below from the dataset pages (they stay empty until you do) and compare them with the measured colour of the mushrooms. Three random photos per farm help to check the colour by eye.

In [ ]:
# 12. Original label names per farm, species notes, measured colour, three random photos per farm
SPECIES_NOTES = {   # fill in from each dataset page (species or colour type); the files do not say
    "duman": "", "angelo": "white oysters (per the run 2 notebook text)", "abhi": "pink oysters on buckets in a grow tent (per the run 1 notebook text; species not stated)",
    "amr": "", "thesis": "", "ros2": "",
}
rows = []
for f in FARMS:
    for name, n in THEIR[f].most_common():
        ours = MAPPING[f].get(norm(name), "dropped (not mapped)")
        rows.append({"farm": f, "their label": name, "boxes": n, "our label": ours})
STAGES = pd.DataFrame(rows)
display(STAGES.set_index(["farm", "their label"]))

colour = PH.groupby("farm")[["rb_in", "S_in", "pink_share"]].median().round(2).loc[FARMS]
colour.columns = ["median R/B in boxes", "median S in boxes", "median % of coloured pixels pink"]
colour.insert(0, "species / colour note (yours)", [SPECIES_NOTES.get(f, "") or "(fill in)" for f in FARMS])
display(colour)

rng = np.random.RandomState(SEED)
groups = {f: [(p, "") for p in rng.choice(PH.loc[PH.farm == f, "path"].values, size=min(3, (PH.farm == f).sum()), replace=False)]
          for f in FARMS}
show_thumbs(groups, "3 random photos per farm (check the mushroom colour by eye)")

## 7. Summary table (screenshot this)
One row per farm. Colour code: none, read the numbers. "farm guessed right %" is from section 5 (colour + size); "background AUC" is from section 4 (young vs ready from the background only; blank = could not be tested).

In [ ]:
# 13. One row per farm
S = pd.DataFrame(index=FARMS)
S["photos"] = [int(INV.loc["photos", f]) for f in FARMS]
S["dup groups"] = [int(INV.loc["near-duplicate groups (2+ photos)", f]) for f in FARMS]
S["boxes/photo"] = [INV.loc["boxes per photo (mean)", f] for f in FARMS]
S["ready %"] = [INV.loc["ready share %", f] for f in FARMS]
S["main size WxH"] = [INV.loc["most common sizes WxH (photos)", f].split(" (")[0] for f in FARMS]
g = PH.groupby("farm")
S["sharpness"] = g.sharp.median().round(0).reindex(FARMS)
S["% blown"] = g.blown.median().round(2).reindex(FARMS)
S["% dark"] = g.dark.median().round(2).reindex(FARMS)
S["mean V"] = g.meanV.mean().round(0).reindex(FARMS)
S["mean S"] = g.meanS.mean().round(0).reindex(FARMS)
S["R/B in boxes"] = g.rb_in.median().round(2).reindex(FARMS)
S["boxes cover %"] = B.groupby("farm").cover.median().round(1).reindex(FARMS)
S["background AUC"] = pd.Series(BG).reindex(FARMS)
S["farm guessed right %"] = pd.Series(GUESS).reindex(FARMS)
S["stages (their labels)"] = [", ".join(THEIR[f]) for f in FARMS]
S.index.name = "farm"
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None); pd.set_option("display.max_colwidth", 60)
if SKIPPED:
    print("SKIPPED sets (not in the table):", SKIPPED)
S

## Read the result
* **Farm guessed right % high** (section 5, and a different `main size`, `mean V`, `R/B`) = the farms look very different; leave-one-farm-out will fail until the training photos cover that look. This is the likely story behind run 2's 0.00.
* **Background AUC clearly above 0.5** (section 4) = young and ready differ by background or camera distance inside that farm, so a within-farm score can be inflated by a shortcut that does not travel to other farms.
* **Quality columns** (`sharpness`, `% blown`, `% dark`) = a farm with much worse values may need its own quality gate or should be handled with care; the thumbnails in section 2 show what that looks like.
* **ready %** near 0 or 100 = nearly one label; the model can score well by guessing.
* **Next.** Use this table to decide the next Kaggle run (for example photos at similar distances, tiled training, or colour augmentation); syldd photos stay a later, separate test.

In [ ]:
# 14. Remove the temporary photo folders
import shutil
for d in glob.glob(f"{WORK}/raw_*") + glob.glob(f"{WORK}/pool_*"):
    shutil.rmtree(d, ignore_errors=True)
print("temporary folders removed")